In [5]:
import matplotlib.pyplot as plt
import os
import numpy as np
# from tqdm import tqdm
from network_funcs import *
from qopt_funcs import *
import networkx as nx
# import random
import time
import shutil    # to copy files at the end of the script

In [31]:
def plot_network(cartesian_coords, rate_matrix, R, filename='earth', caption='',
                         bckgrnd_color = 'midnightblue', pt_color='yellow', edge_color='white'):    # ft chatgpt
    # could be improved, eg R must be the length of the coords vecs so its just messy to input it separately
    deg_distr = np.sum(rate_matrix > 0, axis=1)
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    # generate cartesian coords arrays
    x = cartesian_coords[:, 0]
    y = cartesian_coords[:, 1]
    z = cartesian_coords[:, 2]
    # represent transparent sphere
    u = np.linspace(0, 2 * np.pi, 100)
    v = np.linspace(0, np.pi, 100)
    x_sphere = R * np.outer(np.cos(u), np.sin(v))
    y_sphere = R * np.outer(np.sin(u), np.sin(v))
    z_sphere = R * np.outer(np.ones(np.size(u)), np.cos(v))
    ax.plot_surface(x_sphere, y_sphere, z_sphere, color='blue', alpha=0.2)
    # represent points
    ax.scatter(x, y, z, s=np.log(deg_distr+1)*50, color=pt_color)
    # represent paths between points as geodetic curves
    n_points = cartesian_coords.shape[0]
    for i in range(n_points):
        for j in range(i+1, n_points):
            if rate_matrix[i, j] > 0:
                phi = np.arccos(np.dot(cartesian_coords[i], cartesian_coords[j]) / (R**2))    # the angle bw the 2 vecs
                t = np.linspace(0, phi, 100)               # for the parametric curve
                x_arc = np.sin(t) * (cartesian_coords[i, 0] / np.sin(phi)) + np.sin(phi - t) * (cartesian_coords[j, 0] / np.sin(phi))
                y_arc = np.sin(t) * (cartesian_coords[i, 1] / np.sin(phi)) + np.sin(phi - t) * (cartesian_coords[j, 1] / np.sin(phi))
                z_arc = np.sin(t) * (cartesian_coords[i, 2] / np.sin(phi)) + np.sin(phi - t) * (cartesian_coords[j, 2] / np.sin(phi))
                ax.plot(x_arc, y_arc, z_arc, color=edge_color, alpha=0.5, linewidth=2./(1. - np.log(rate_matrix[i, j])))
    # require same scale for all axes
    ax.set_xlim([-R, R])
    ax.set_ylim([-R, R])
    ax.set_zlim([-R, R])
    ax.set_box_aspect([1, 1, 1])
    ax.axis('off')
    #plt.rcParams['figure.figsize'] = [15,15]
    fig.set_size_inches(15,15)
    ax.set_facecolor(bckgrnd_color)
    #plt.show()            # interactive 3d mode (not compatible w savefig)
    fig.text(.5, .15, s=caption, color=edge_color, fontsize=30)
    output_dir = 'earth_frames'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    if not os.path.exists(output_dir + '/' + bckgrnd_color):
        os.makedirs(output_dir + '/' + bckgrnd_color)
    filename = 'earth_frames/' + bckgrnd_color + '/' + filename + '.png'
    plt.savefig(filename, dpi=300, transparent=True)
    plt.close(fig)
    return


In [32]:
beta = 2.6261                 # \beta param of S2 model
mu = 0.0233                   # \mu param of S2 model
N=100

rhos = 0.14*10**np.linspace(-2.3,-1.9,3)
radii = np.sqrt(N/4/np.pi/rhos)

# for it in range(n_iter):    # indent back below for multiple iterations

A, Dists, coords = S2_graph_definite_N(N, beta, mu, return_coords=True)
n_nodes_giant = []                                   # nr of nodes in largest (giant) component
clustering_list = []
rate_sum_accum = []
rate_sq_sum_accum = []
for radius in radii:
    T = np.zeros_like(A)                    # matrix of the graph weights ie the inverse rates (s/bit)
    for i in range(N):
        for j in range(i):
            if A[i,j] == 1:
                dij = radius*Dists[i,j]
                T[i,j] = T[j,i] = transmittance_v_distance(dij)
    #plot_graph_on_sphere(coords, A_pruned, 1, filename='earth_radius%.2f' % radius)    # makes things slower
    G_pruned = nx.from_numpy_array(T)          # return a weighted graph object

    plot_network(coords, T, 1, filename='earth_radius%.2f' % radius)    # makes things slower
    plt.show()
